# Dallas College Success Coach Chatbot - Chunking & Embedding Pipeline

**Project**: Dallas College Success Coach Chatbot (Antigravity Club)

## What This Notebook Does

This notebook takes **raw scraped data** from the Dallas College catalog and turns it into a
**searchable vector database** that our chatbot can query.

### The Pipeline
```
Scraped JSON Files
  -> Chunking (split long text into smaller pieces)
    -> Embedding (convert text to numerical vectors)
      -> ChromaDB (store vectors for fast similarity search)
        -> Chatbot can now find relevant info!
```

### Key Concepts

- **Chunking**: Large documents are split into smaller pieces (500-1000 tokens).
  Language models have context limits, and smaller chunks give more precise search results.

- **Embedding**: Each chunk is converted into a vector (a list of numbers) that captures its
  *meaning*. Similar text produces similar vectors. We use the `all-MiniLM-L6-v2` model,
  which outputs 384-dimensional vectors.

- **ChromaDB**: An open-source vector database. It stores our embeddings and lets us do
  *similarity search* -- given a user question, find the most relevant chunks of catalog data.

---

## Cell 1: Setup & Install Dependencies

In [ ]:
# =============================================================================
# CELL 1: Setup & Install
# =============================================================================
# We need two main libraries:
#
# 1. chromadb - Our vector database. Think of it like a search engine that
#    understands *meaning*, not just keyword matching.
#    - It stores text + embeddings (vector representations of text)
#    - It uses HNSW (Hierarchical Navigable Small World) indexing for fast
#      approximate nearest-neighbor search
#    - It can persist to disk so we don't lose our data between sessions
#    - It's open-source and works well for projects our size (<1M documents)
#
# 2. sentence-transformers - Provides pre-trained models that convert text
#    into embedding vectors. The default model (all-MiniLM-L6-v2) is:
#    - Fast (good for a Colab notebook)
#    - Produces 384-dimensional vectors
#    - Good quality for English semantic search tasks
#
# Why ChromaDB over alternatives?
# - Pinecone/Weaviate: cloud-hosted, cost money at scale
# - FAISS: lower-level, no built-in metadata filtering
# - ChromaDB: free, easy API, built-in metadata, persistent storage
# =============================================================================

!pip install -q chromadb sentence-transformers

import json
import os
import re
import math
import shutil
import zipfile
from pathlib import Path
from datetime import datetime

import chromadb
from chromadb.utils import embedding_functions

print(f"chromadb version: {chromadb.__version__}")
print("Setup complete!")

## Cell 2: Load Scraped Data

We have three types of scraped data:

| File | What it contains | Example fields |
|------|-----------------|----------------|
| `courses.json` | Every course in the catalog | prefix, number, title, description, prerequisites |
| `programs.json` | Degree/certificate programs | title, degree_type, description, requirements |
| `general_pages.json` | Info pages (transfer, financial aid, etc.) | category, label, content |

The notebook tries to load from **Google Drive first** (if you're running in Colab with
Drive mounted), then falls back to letting you **upload files manually**.

In [ ]:
# =============================================================================
# CELL 2: Upload / Load Data
# =============================================================================
# Two loading modes:
#   Mode A: Google Drive (Colab with Drive mounted)
#   Mode B: File upload (fallback for any environment)
#
# The scraped data has three files:
#   - courses.json: Array of course objects with prefix, number, title,
#     description, prerequisites, etc.
#   - programs.json: Array of degree/certificate program objects with
#     title, degree_type, description, requirements, etc.
#   - general_pages.json: Array of general info pages with category,
#     label, content, url, etc.
# =============================================================================

# ---------- Configuration ----------
# Path where scraped data lives on Google Drive (when mounted in Colab)
GDRIVE_DATA_PATH = (
    "/content/drive/MyDrive/AntigravityProjects/"
    "SucesscoachChatbot/apps/data/scraped/"
)

# Local fallback path (for running outside Colab)
LOCAL_DATA_PATH = "./scraped/"

DATA_FILES = ["courses.json", "programs.json", "general_pages.json"]


def load_json(filepath: str) -> list:
    """Load a JSON file and return its contents (expected to be a list)."""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"  Loaded {filepath}: {len(data)} records")
        return data
    except FileNotFoundError:
        print(f"  WARNING: {filepath} not found, skipping.")
        return []
    except json.JSONDecodeError as e:
        print(f"  ERROR: {filepath} is not valid JSON: {e}")
        return []


def try_mount_gdrive() -> bool:
    """Attempt to mount Google Drive. Returns True if successful."""
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return True
    except ImportError:
        print("Not running in Colab -- skipping Drive mount.")
        return False
    except Exception as e:
        print(f"Drive mount failed: {e}")
        return False


def upload_files() -> str:
    """Upload files via Colab widget. Returns the directory they were saved to."""
    try:
        from google.colab import files
        print("Please upload courses.json, programs.json, and general_pages.json:")
        uploaded = files.upload()
        upload_dir = "./uploaded_data/"
        os.makedirs(upload_dir, exist_ok=True)
        for filename, content in uploaded.items():
            with open(os.path.join(upload_dir, filename), "wb") as f:
                f.write(content)
        return upload_dir
    except ImportError:
        print("File upload not available outside Colab.")
        return LOCAL_DATA_PATH


# ---------- Load data ----------
data_path = None

# Mode A: Try Google Drive
if os.path.exists(GDRIVE_DATA_PATH):
    print("Found data on Google Drive (already mounted).")
    data_path = GDRIVE_DATA_PATH
else:
    mounted = try_mount_gdrive()
    if mounted and os.path.exists(GDRIVE_DATA_PATH):
        print("Loaded data from Google Drive.")
        data_path = GDRIVE_DATA_PATH

# Mode B: Try local path
if data_path is None and os.path.exists(LOCAL_DATA_PATH):
    print(f"Found data at local path: {LOCAL_DATA_PATH}")
    data_path = LOCAL_DATA_PATH

# Mode C: Upload fallback
if data_path is None:
    print("Data not found on Drive or locally. Falling back to file upload.")
    data_path = upload_files()

# Load each file
print(f"\nLoading data from: {data_path}")
courses = load_json(os.path.join(data_path, "courses.json"))
programs = load_json(os.path.join(data_path, "programs.json"))
general_pages = load_json(os.path.join(data_path, "general_pages.json"))

total_records = len(courses) + len(programs) + len(general_pages)
print(f"\nTotal records loaded: {total_records}")
print(f"  Courses:       {len(courses)}")
print(f"  Programs:      {len(programs)}")
print(f"  General Pages: {len(general_pages)}")

if total_records == 0:
    raise ValueError(
        "No data loaded! Check that the JSON files exist and are non-empty."
    )

## Cell 3: Chunking Logic

### Why do we chunk?

Large documents need to be split into smaller pieces for two reasons:

1. **Precision**: If a user asks about prerequisites for COSC 1436, we want to return
   *just* the relevant section, not a 5000-word page.

2. **Embedding quality**: Embedding models work best on shorter text (a few sentences
   to a paragraph). Longer text gets "averaged out" and loses specificity.

### Our chunking strategy

- **Target size**: 500-1000 tokens per chunk (~375-750 words)
- **Overlap**: 10% of chunk size. This means the end of one chunk overlaps with the
  beginning of the next. This prevents information from being split across chunk
  boundaries and lost.
- **Metadata**: Each chunk carries metadata (source type, title, URL, etc.) so we
  know where the information came from.

```
Original text: [AAAA BBBB CCCC DDDD EEEE FFFF GGGG HHHH]

Chunk 1: [AAAA BBBB CCCC DDDD]
Chunk 2:           [CCCC DDDD EEEE FFFF]    <- overlaps with chunk 1
Chunk 3:                     [EEEE FFFF GGGG HHHH]
```

In [ ]:
# =============================================================================
# CELL 3: Chunking Logic
# =============================================================================
# We implement a simple but effective token-based chunking strategy.
#
# "Token" here means whitespace-separated words. This is a rough approximation
# -- real tokenizers (like tiktoken for GPT) produce different counts -- but
# it's close enough for chunking purposes and keeps our code simple.
#
# Parameters:
#   CHUNK_SIZE: Target number of tokens per chunk (500-1000)
#   OVERLAP_FRACTION: What fraction of chunk_size to overlap (0.10 = 10%)
# =============================================================================

CHUNK_SIZE = 750          # tokens per chunk (target)
OVERLAP_FRACTION = 0.10   # 10% overlap between consecutive chunks
MIN_CHUNK_SIZE = 50       # don't create tiny chunks


def estimate_tokens(text: str) -> int:
    """Rough token count by splitting on whitespace."""
    return len(text.split())


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap_fraction: float = OVERLAP_FRACTION) -> list[str]:
    """
    Split text into overlapping chunks of approximately `chunk_size` tokens.

    Returns a list of text chunks. If the text is shorter than chunk_size,
    returns it as a single chunk.
    """
    words = text.split()
    if len(words) <= chunk_size:
        return [text]

    overlap = int(chunk_size * overlap_fraction)  # number of overlapping tokens
    step = chunk_size - overlap                    # how far to advance each step

    chunks = []
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]

        # Skip tiny trailing chunks
        if len(chunk_words) < MIN_CHUNK_SIZE and len(chunks) > 0:
            # Append remainder to last chunk instead of creating a tiny one
            chunks[-1] = chunks[-1] + " " + " ".join(chunk_words)
            break

        chunks.append(" ".join(chunk_words))

        if end >= len(words):
            break

    return chunks


def clean_text(text: str) -> str:
    """Basic text cleaning: collapse whitespace, strip BOM and control chars."""
    if not text:
        return ""
    # Remove BOM and zero-width chars
    text = text.replace("\ufeff", "").replace("\u200b", "")
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ---------- Build chunks from each data source ----------

all_chunks = []  # Each element: {"id": str, "text": str, "metadata": dict}


def add_chunks(text: str, source_type: str, source_id: str,
               metadata: dict):
    """
    Chunk the text and add each chunk to all_chunks with metadata.
    """
    text = clean_text(text)
    if not text or len(text.split()) < 10:
        return  # skip empty/trivial content

    chunks = chunk_text(text)
    for i, chunk in enumerate(chunks):
        chunk_id = f"{source_type}_{source_id}_chunk{i}"
        chunk_meta = {
            "source_type": source_type,
            "source_id": str(source_id),
            "chunk_index": i,
            "total_chunks": len(chunks),
            **metadata,  # merge in type-specific metadata
        }
        # ChromaDB metadata values must be str, int, float, or bool
        chunk_meta = {
            k: (str(v) if v is not None else "")
            for k, v in chunk_meta.items()
        }
        all_chunks.append({
            "id": chunk_id,
            "text": chunk,
            "metadata": chunk_meta,
        })


# --- Courses ---
# For courses, the main text is the description. We also prepend a header
# line so the embedding captures the course identity.
for course in courses:
    header = f"{course.get('prefix', '')} {course.get('number', '')} - {course.get('title', '')}"
    parts = [header]
    if course.get("description"):
        parts.append(course["description"])
    if course.get("prerequisites"):
        parts.append(f"Prerequisites: {course['prerequisites']}")
    if course.get("corequisites"):
        parts.append(f"Corequisites: {course['corequisites']}")

    full_text = "\n".join(parts)

    add_chunks(
        text=full_text,
        source_type="course",
        source_id=str(course.get("coid", "")),
        metadata={
            "title": course.get("title", ""),
            "prefix": course.get("prefix", ""),
            "number": course.get("number", ""),
            "credits": course.get("credits", ""),
            "url": course.get("url", ""),
        },
    )

# --- Programs ---
for program in programs:
    parts = [program.get("title", "")]
    if program.get("description"):
        parts.append(program["description"])
    if program.get("requirements"):
        parts.append(f"Requirements:\n{program['requirements']}")

    full_text = "\n".join(parts)

    add_chunks(
        text=full_text,
        source_type="program",
        source_id=str(program.get("poid", "")),
        metadata={
            "title": program.get("title", ""),
            "degree_type": program.get("degree_type", ""),
            "url": program.get("url", ""),
        },
    )

# --- General Pages ---
for page in general_pages:
    parts = [page.get("label", "")]
    if page.get("content"):
        parts.append(page["content"])

    full_text = "\n".join(parts)

    add_chunks(
        text=full_text,
        source_type="general",
        source_id=str(page.get("navoid", "")),
        metadata={
            "title": page.get("label", ""),
            "category": page.get("category", ""),
            "url": page.get("url", ""),
        },
    )


# ---------- Summary ----------
print(f"Total chunks created: {len(all_chunks)}")

# Count by source type
from collections import Counter
type_counts = Counter(c["metadata"]["source_type"] for c in all_chunks)
for stype, count in sorted(type_counts.items()):
    print(f"  {stype}: {count} chunks")

# Show a sample chunk
if all_chunks:
    sample = all_chunks[0]
    print(f"\n--- Sample Chunk ---")
    print(f"ID: {sample['id']}")
    print(f"Text (first 200 chars): {sample['text'][:200]}...")
    print(f"Metadata: {sample['metadata']}")

## Cell 4: ChromaDB Collection Setup

### How ChromaDB Works

ChromaDB stores documents in **collections**. Each collection has:
- **Documents**: The actual text chunks
- **Embeddings**: Vector representations (computed automatically by the embedding function)
- **Metadata**: Key-value pairs attached to each document (we use these for filtering)
- **IDs**: Unique identifiers for each document

### HNSW Index

Under the hood, ChromaDB uses an **HNSW (Hierarchical Navigable Small World)** index.
Think of it as a graph where similar items are connected. When you search, it navigates
this graph to find the nearest neighbors quickly -- much faster than comparing against
every document (which would be O(n)).

Key HNSW parameters:
- `hnsw:space` = "cosine" -- we measure similarity using cosine distance
- `hnsw:M` = 16 -- number of connections per node (higher = more accurate but slower)
- `hnsw:construction_ef` = 100 -- search width during index building

In [ ]:
# =============================================================================
# CELL 4: ChromaDB Collection Setup
# =============================================================================
# We create a *persistent* ChromaDB client. This means the database is saved
# to disk (in ./chroma_db/) and survives kernel restarts.
#
# Embedding function:
#   We use sentence-transformers' all-MiniLM-L6-v2 model. It's the default in
#   ChromaDB and is a good balance of speed and quality.
#
#   To swap in a different model (e.g., for better quality), change the
#   model_name parameter below. Some alternatives:
#     - "all-mpnet-base-v2" (higher quality, slower)
#     - "paraphrase-MiniLM-L3-v2" (faster, lower quality)
#     - Or use OpenAI/Cohere embeddings via their ChromaDB integrations
#
# HNSW index configuration:
#   - "hnsw:space": "cosine" -- Cosine similarity is standard for text.
#     It measures the angle between vectors, ignoring magnitude.
#     Cosine distance = 1 - cosine_similarity, so LOWER distance = MORE similar.
#   - "hnsw:M": 16 -- Each node connects to 16 neighbors. This is the default
#     and works well for most datasets. Increase for higher recall at the
#     cost of more memory.
#   - "hnsw:construction_ef": 100 -- Controls how thoroughly the index is
#     built. Higher = better quality index, slower build.
# =============================================================================

CHROMA_DB_PATH = "./chroma_db"
COLLECTION_NAME = "dallas_college_catalog"

# --- Embedding function ---
# Change model_name here to swap in a different embedding model.
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# --- Create persistent client ---
# PersistentClient saves the database to disk automatically.
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# --- Create (or get) collection ---
# get_or_create_collection: if it already exists, reuse it; otherwise create it.
# We delete any existing collection first to ensure a clean build.
try:
    client.delete_collection(name=COLLECTION_NAME)
    print(f"Deleted existing collection '{COLLECTION_NAME}' for clean rebuild.")
except ValueError:
    pass  # Collection didn't exist, that's fine

collection = client.create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={
        "hnsw:space": "cosine",          # cosine similarity
        "hnsw:M": 16,                     # connections per node
        "hnsw:construction_ef": 100,       # build-time search width
        "description": "Dallas College catalog data for Success Coach chatbot",
    },
)

print(f"Collection '{COLLECTION_NAME}' created.")
print(f"  Embedding model: all-MiniLM-L6-v2 (384 dimensions)")
print(f"  Distance metric: cosine")
print(f"  Storage path: {CHROMA_DB_PATH}")

## Cell 5: Embed & Store Chunks in ChromaDB

This is where the magic happens. We feed our text chunks into ChromaDB, which:
1. Passes each chunk through the embedding model to get a vector
2. Indexes the vector in the HNSW graph
3. Stores the original text and metadata alongside the vector

We insert in **batches of 100** to avoid memory issues on Colab's free tier.
ChromaDB computes the embeddings for us since we set up the embedding function
on the collection.

In [ ]:
# =============================================================================
# CELL 5: Embed & Store
# =============================================================================
# Batch insertion avoids OOM (out-of-memory) errors.
# ChromaDB automatically computes embeddings using the embedding function
# we attached to the collection in Cell 4.
#
# Each chunk is stored with:
#   - id: unique identifier (e.g., "course_15128_chunk0")
#   - document: the text content
#   - metadata: source_type, title, url, etc.
# =============================================================================

BATCH_SIZE = 100
total = len(all_chunks)
num_batches = math.ceil(total / BATCH_SIZE)

print(f"Inserting {total} chunks in {num_batches} batches of {BATCH_SIZE}...")
print()

for batch_idx in range(num_batches):
    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, total)
    batch = all_chunks[start:end]

    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[c["metadata"] for c in batch],
    )

    # Progress indicator
    pct = ((batch_idx + 1) / num_batches) * 100
    bar = "#" * int(pct // 2) + "-" * (50 - int(pct // 2))
    print(f"  [{bar}] {pct:5.1f}%  ({end}/{total} chunks)", end="\r")

print()  # newline after progress bar
print(f"\nDone! Collection now has {collection.count()} documents.")

## Cell 6: Test Similarity Searches

Now let's test the database with some realistic student questions. ChromaDB will:
1. Embed the query using the same model
2. Find the closest vectors in the HNSW index
3. Return the matching documents with their distance scores

**Distance interpretation** (cosine):
- 0.0 = identical
- 0.0 - 0.3 = very similar (good match)
- 0.3 - 0.6 = somewhat related
- 0.6+ = not very related

In [ ]:
# =============================================================================
# CELL 6: Test Queries
# =============================================================================
# We run a few sample queries that a student might ask the chatbot.
# For each query, we retrieve the top 3 most similar chunks.
#
# The `query` method:
#   - Embeds the query text using the same model
#   - Finds the nearest neighbors in vector space
#   - Returns documents, metadata, and distances
#
# Note: "distances" are cosine distances (1 - cosine_similarity).
# Lower distance = more similar.
# =============================================================================

test_queries = [
    "What are the prerequisites for COSC 1436?",
    "Where is the success coaching office at Richland?",
    "How do I transfer to UT Dallas?",
    "What financial aid deadlines should I know?",
]

N_RESULTS = 3  # how many results to return per query

for query in test_queries:
    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)

    results = collection.query(
        query_texts=[query],
        n_results=N_RESULTS,
        include=["documents", "metadatas", "distances"],
    )

    if not results["documents"][0]:
        print("  No results found.\n")
        continue

    for i in range(len(results["documents"][0])):
        doc = results["documents"][0][i]
        meta = results["metadatas"][0][i]
        dist = results["distances"][0][i]
        similarity = 1 - dist  # convert distance to similarity score

        print(f"\n  Result {i + 1} (similarity: {similarity:.3f}, distance: {dist:.3f})")
        print(f"  Source: {meta.get('source_type', '?')} | {meta.get('title', 'N/A')}")
        print(f"  URL: {meta.get('url', 'N/A')}")
        if meta.get("prefix"):
            print(f"  Course: {meta.get('prefix', '')} {meta.get('number', '')} ({meta.get('credits', '?')} credits)")
        if meta.get("degree_type"):
            print(f"  Degree: {meta.get('degree_type', '')}")
        if meta.get("category"):
            print(f"  Category: {meta.get('category', '')}")
        # Show first 300 chars of the document
        print(f"  Text preview: {doc[:300]}...")

    print()

## Cell 7: Export

We export two things:
1. **`chunked_data.json`** -- All chunks with their metadata (useful for debugging or
   loading into other systems)
2. **`chroma_db.zip`** -- The entire ChromaDB database directory, zipped for download
   or backup. You can unzip this and point a ChromaDB PersistentClient at it to
   restore the database.

In [ ]:
# =============================================================================
# CELL 7: Export
# =============================================================================
# Export the chunked data and the ChromaDB database for backup/download.
# =============================================================================

# --- Export chunked_data.json ---
CHUNKED_DATA_PATH = "./chunked_data.json"

with open(CHUNKED_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

file_size_mb = os.path.getsize(CHUNKED_DATA_PATH) / (1024 * 1024)
print(f"Exported {len(all_chunks)} chunks to {CHUNKED_DATA_PATH} ({file_size_mb:.2f} MB)")


# --- Zip the ChromaDB directory ---
ZIP_PATH = "./chroma_db.zip"

# Remove old zip if it exists
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(CHROMA_DB_PATH):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(CHROMA_DB_PATH))
            zf.write(file_path, arcname)

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"Exported ChromaDB to {ZIP_PATH} ({zip_size_mb:.2f} MB)")

# --- Colab download (if available) ---
try:
    from google.colab import files
    print("\nDownloading files...")
    files.download(CHUNKED_DATA_PATH)
    files.download(ZIP_PATH)
except ImportError:
    print(f"\nNot in Colab. Files saved locally:")
    print(f"  {os.path.abspath(CHUNKED_DATA_PATH)}")
    print(f"  {os.path.abspath(ZIP_PATH)}")

## Cell 8: Summary Statistics

A quick overview of what we built.

In [ ]:
# =============================================================================
# CELL 8: Stats
# =============================================================================
# Print summary statistics about the chunked and embedded data.
# =============================================================================

from collections import Counter

print("=" * 60)
print("  DALLAS COLLEGE CATALOG - EMBEDDING PIPELINE SUMMARY")
print("=" * 60)
print()

# --- Source data ---
print("SOURCE DATA")
print(f"  Courses:       {len(courses):>6}")
print(f"  Programs:      {len(programs):>6}")
print(f"  General Pages: {len(general_pages):>6}")
print(f"  Total Records: {len(courses) + len(programs) + len(general_pages):>6}")
print()

# --- Chunks ---
print("CHUNKS")
print(f"  Total chunks: {len(all_chunks)}")

type_counts = Counter(c["metadata"]["source_type"] for c in all_chunks)
for stype, count in sorted(type_counts.items()):
    print(f"    {stype}: {count}")

chunk_sizes = [len(c["text"].split()) for c in all_chunks]
if chunk_sizes:
    print(f"  Avg chunk size: {sum(chunk_sizes) / len(chunk_sizes):.0f} tokens")
    print(f"  Min chunk size: {min(chunk_sizes)} tokens")
    print(f"  Max chunk size: {max(chunk_sizes)} tokens")
print()

# --- ChromaDB Collection ---
print("CHROMADB COLLECTION")
print(f"  Name: {COLLECTION_NAME}")
print(f"  Documents stored: {collection.count()}")
print(f"  Embedding model: all-MiniLM-L6-v2 (384 dims)")
print(f"  Distance metric: cosine")
print(f"  Storage: {CHROMA_DB_PATH}")
print()

# --- Chunking config ---
print("CHUNKING CONFIG")
print(f"  Chunk size: {CHUNK_SIZE} tokens")
print(f"  Overlap: {OVERLAP_FRACTION * 100:.0f}%")
print(f"  Min chunk size: {MIN_CHUNK_SIZE} tokens")
print()

print(f"Pipeline completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)